In [13]:
pip install torch-geometric

Defaulting to user installation because normal site-packages is not writeable
  Using cached pyparsing-3.3.2-py3-none-any.whl.metadata (5.8 kB)
  Using cached aiosignal-1.4.0-py3-none-any.whl.metadata (3.7 kB)
   ---------------------------------------- 0.0/1.3 MB ? eta -:--:--
   ------------------------------- -------- 1.0/1.3 MB 7.9 MB/s eta 0:00:01
   ------------------------------- -------- 1.0/1.3 MB 7.9 MB/s eta 0:00:01
   ---------------------------------------- 1.3/1.3 MB 2.4 MB/s eta 0:00:00
Using cached aiosignal-1.4.0-py3-none-any.whl (7.5 kB)
Using cached pyparsing-3.3.2-py3-none-any.whl (122 kB)

   ---- -----------------------------------  1/10 [pyparsing]
   ---- -----------------------------------  1/10 [pyparsing]
   -------- -------------------------------  2/10 [propcache]
   ---------------- -----------------------  4/10 [frozenlist]
   ------------------------ ---------------  6/10 [yarl]
   ------------------------ ---------------  6/10 [yarl]
   ----------------


[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [14]:
import torch
import cv2
from PIL import Image
from torchvision import transforms
from torch.utils.data import Dataset
from torch_geometric.data import Data
from sklearn.metrics.pairwise import cosine_similarity

C:\Users\len\AppData\Roaming\Python\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [15]:
import zipfile
import os

with zipfile.ZipFile("split_frames_dataset (1).zip", "r") as zip_ref:
    zip_ref.extractall("split_frames_dataset")

# check extracted folder
print(os.listdir("."))

['.git', 'code.ipynb', 'deepfake_detection (1).zip', 'README.md', 'split_frames_dataset', 'split_frames_dataset (1).zip']


In [16]:
import glob
import os

def load_image_paths(base_dir, label):
    files = glob.glob(os.path.join(base_dir, "**", "*.jpg"), recursive=True)
    return [(f, label) for f in files]

# Train
real_train_files = load_image_paths("split_frames_dataset/split_frames_dataset/train/real", 0)
fake_train_files = load_image_paths("split_frames_dataset/split_frames_dataset/train/fake", 1)

# Test
real_val_files = load_image_paths("split_frames_dataset/split_frames_dataset/test/real", 0)
fake_val_files = load_image_paths("split_frames_dataset/split_frames_dataset/test/fake", 1)

print("✅ Train Real:", len(real_train_files))
print("✅ Train Fake:", len(fake_train_files))
print("✅ Test Real:", len(real_val_files))
print("✅ Test Fake:", len(fake_val_files))

✅ Train Real: 166
✅ Train Fake: 166
✅ Test Real: 42
✅ Test Fake: 42


In [17]:
def image_to_graph(image_tensor, k=9, patch_size=32, debug=True):
    C, H, W = image_tensor.shape

    if H < patch_size or W < patch_size:
        if debug:
            print("Skipping tiny frame")
        return None

    patches = image_tensor.unfold(1, patch_size, patch_size).unfold(2, patch_size, patch_size)
    patches = patches.permute(1, 2, 0, 3, 4).contiguous()
    patches = patches.view(-1, C * patch_size * patch_size)

    if patches.size(0) < 2:
        if debug:
            print("Too few patches")
        return None

    similarity = cosine_similarity(patches.cpu().numpy())

    edge_index = []
    for i in range(len(patches)):
        indices = similarity[i].argsort()[-k-1:-1]
        edge_index += [(i, j) for j in indices]

    if len(edge_index) == 0:
        if debug:
            print("No edges created")
        return None

    edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()
    x = patches.float()

    return Data(x=x, edge_index=edge_index)

In [18]:
class FaceFusionDataset(Dataset):
    def __init__(self, file_list):
        self.file_list = file_list
        self.transform = transforms.Compose([
            transforms.Resize((128, 128)),
            transforms.ToTensor()
        ])

    def __len__(self):
        return len(self.file_list)

    def __getitem__(self, idx):
        img_path, label = self.file_list[idx]

        attempts = 0
        while attempts < len(self.file_list):
            image = cv2.imread(img_path)

            if image is None:
                idx = (idx + 1) % len(self.file_list)
                attempts += 1
                continue

            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
            image_tensor = self.transform(Image.fromarray(image))

            graph = image_to_graph(image_tensor)

            if graph is not None:
                graph.y = torch.tensor(label, dtype=torch.long)
                return image_tensor, graph

            idx = (idx + 1) % len(self.file_list)
            attempts += 1

        raise RuntimeError("❌ All images failed to generate a valid graph.")

In [20]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models

from torch_geometric.nn import GCNConv, global_max_pool

In [21]:
# Define Model Architectures
#code along here
import torchvision.models as models

class CNNBranch(nn.Module):
  def __init__(self):
    super().__init__()
    base = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
    self.feature_extractor = nn.Sequential(*list(base.children())[:-1])
    self.fc = nn.Linear(512, 600)

  def forward(self,x):
    x = self.feature_extractor(x).view(x.size(0),-1)
    return self.fc(x)

class HierarchicalGNNBranch(nn.Module):
  def __init__(self):
    super().__init__() # Added this line
    self.fds = [80,160,400,600]
    self.mlps = nn.ModuleList([
        nn.Sequential(
            nn.Linear(3072 if i==0 else self.fds[i-1],fd*4),
            nn.ReLU(),
            nn.Dropout(0.6),
            nn.Linear(fd*4,fd),
            nn.ReLU(),
            nn.Dropout(0.6)
        ) for i,fd in enumerate(self.fds)
    ])
    self.convs = nn.ModuleList([GCNConv(fd,fd) for fd in self.fds])
    self.norms = nn.ModuleList([nn.BatchNorm1d(fd) for fd in self.fds])
    self.dropout = nn.Dropout(0.6)

  def forward(self,data):
    x,edge_index,batch = data.x, data.edge_index, data.batch
    for i in range(len(self.fds)):
      x_res = x # Store the output of the previous layer before applying current MLP
      x = self.mlps[i](x)
      x = self.convs[i](x,edge_index)
      x = self.norms[i](x)
      if i > 0 and x.size(1) == x_res.size(1): # Apply residual connection only after the first layer AND if feature sizes match
          x = F.relu(x+x_res)
      else:
          x = F.relu(x) # Just apply ReLU otherwise
      x = self.dropout(x)
    return global_max_pool(x,batch)

class FuNetA(nn.Module):
  def __init__(self):
    super().__init__()
    self.cnn = CNNBranch()
    self.gnn = HierarchicalGNNBranch()
    self.fc = nn.Linear(600,2)
  def forward(self,img,graph):
    return self.fc(self.cnn(img)+self.gnn(graph))

class FuNetM(nn.Module):
  def __init__(self):
    super().__init__()
    self.cnn = CNNBranch()
    self.gnn = HierarchicalGNNBranch()
    self.fc = nn.Linear(600,2)
  def forward(self,img,graph):
    return self.fc(self.cnn(img)*self.gnn(graph))


class FuNetC(nn.Module):
  def __init__(self):
    super().__init__()
    self.cnn = CNNBranch()
    self.gnn = HierarchicalGNNBranch()
    self.fc = nn.Linear(1200,2)
  def forward(self,img,graph):
    return self.fc(torch.cat([self.cnn(img),self.gnn(graph)],dim=1))

In [22]:
from torch_geometric.data import Batch
from torch.utils.data import DataLoader

# Build train/val datasets
train_set = FaceFusionDataset(real_train_files + fake_train_files)
val_set   = FaceFusionDataset(real_val_files + fake_val_files)

# Custom collate function
def collate_fn(batch):
    batch = [b for b in batch if b is not None]  # filter out None
    if len(batch) == 0:
        return None, None
    images = torch.stack([b[0] for b in batch])
    graphs = Batch.from_data_list([b[1] for b in batch])
    return images, graphs


# DataLoaders
train_loader = DataLoader(train_set, batch_size=8, shuffle=True, collate_fn=collate_fn)
val_loader   = DataLoader(val_set, batch_size=8, shuffle=False, collate_fn=collate_fn)

print("✅ Train samples:", len(train_set))
print("✅ Val samples:", len(val_set))


✅ Train samples: 332
✅ Val samples: 84


In [23]:
from torch_geometric.data import Batch
from torch.utils.data import DataLoader

def train(model, loader, optimizer, device):
    model.train()
    total_loss = 0
    for images, graphs in loader:
        if images is None or graphs is None:
            continue
        images = images.to(device)
        # Explicitly move graph components to device
        graphs.x = graphs.x.to(device)
        graphs.edge_index = graphs.edge_index.to(device)
        if graphs.batch is not None:
            graphs.batch = graphs.batch.to(device)
        graphs.y = graphs.y.to(device) # Move labels to device
        graphs = graphs.to(device) # Move the Batch object itself (might be redundant but keeps the object type)

        optimizer.zero_grad()
        out = model(images, graphs)
        loss = F.cross_entropy(out, graphs.y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

def validate(model, loader, device):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for images, graphs in loader:
            if images is None or graphs is None:
                continue
            images = images.to(device)
            # Explicitly move graph components to device
            graphs.x = graphs.x.to(device)
            graphs.edge_index = graphs.edge_index.to(device)
            if graphs.batch is not None:
                graphs.batch = graphs.batch.to(device)
            graphs.y = graphs.y.to(device) # Move labels to device
            graphs = graphs.to(device) # Move the Batch object itself

            out = model(images, graphs)
            loss = F.cross_entropy(out, graphs.y)
            total_loss += loss.item()
    return total_loss / len(loader)

def evaluate(model, loader, device):
    model.eval()
    y_true, y_pred, y_prob = [], [], []
    with torch.no_grad():
        for images, graphs in loader:
            if images is None or graphs is None:
                continue
            images = images.to(device)
            # Explicitly move graph components to device
            graphs.x = graphs.x.to(device)
            graphs.edge_index = graphs.edge_index.to(device)
            if graphs.batch is not None:
                graphs.batch = graphs.batch.to(device)
            graphs.y = graphs.y.to(device) # Move labels to device
            graphs = graphs.to(device) # Move the Batch object itself

            logits = model(images, graphs)
            probs = F.softmax(logits, dim=1)[:, 1]
            preds = torch.argmax(logits, dim=1)
            y_true.extend(graphs.y.cpu().numpy())
            y_pred.extend(preds.cpu().numpy())
            y_prob.extend(probs.cpu().numpy())
    return {
        'accuracy': accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred),
        'recall': recall_score(y_true, y_pred),
        'f1': f1_score(y_true, y_pred),
        'auc': roc_auc_score(y_true, y_prob)
    }

class EarlyStopping:
    def __init__(self, patience=5, delta=0.0001):
        self.patience = patience
        self.delta = delta
        self.counter = 0
        self.best_loss = float('inf')
        self.early_stop = False
        self.best_model_state = None

    def __call__(self, val_loss, model):
        if val_loss < self.best_loss - self.delta:
            self.best_loss = val_loss
            self.counter = 0
            self.best_model_state = model.state_dict()
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True

In [25]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

In [29]:
#training loop + validation loop + results
#code along
defined_models={
    #'FuNet-A':FuNetA(),
    'FuNet-M':FuNetM(),
    #'FuNet-C':FuNetC()

}
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model_histories={}
results={}
for name,model in defined_models.items():
  model=model.to(device)
  optimizer=torch.optim.AdamW(model.parameters(),lr=2e-4)
  early_stopper=EarlyStopping(patience=5)

  # Initialize best_model_state with the initial model state
  early_stopper.best_model_state = model.state_dict()

  train_losses=[]
  val_losses=[]
  for epoch in range(1,50):
    train_loss=train(model,train_loader,optimizer,device)
    val_loss=validate(model,val_loader,device)
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    early_stopper(val_loss,model)
    print(f"Epoch:{epoch} : Train Loss={train_loss:.4f}, Validation Loss={val_loss:.4f}")
    if early_stopper.early_stop:
      print("Early stopping")
      break

  # Load the best model state
  model.load_state_dict(early_stopper.best_model_state)
  model_histories[name]={
      'train_losses':train_losses,
      'val_losses':val_losses
  }
  results[name]=evaluate(model,val_loader,device)

Epoch:1 : Train Loss=0.8918, Validation Loss=0.4135
Epoch:2 : Train Loss=0.3830, Validation Loss=0.2194
Epoch:3 : Train Loss=0.0600, Validation Loss=0.1495
Epoch:4 : Train Loss=0.0657, Validation Loss=0.1277
Epoch:5 : Train Loss=0.1497, Validation Loss=0.2055
Epoch:6 : Train Loss=0.2473, Validation Loss=0.2374
Epoch:7 : Train Loss=0.2202, Validation Loss=0.3277
Epoch:8 : Train Loss=0.0643, Validation Loss=0.3248
Epoch:9 : Train Loss=0.0566, Validation Loss=0.1296
Early stopping


In [30]:
for name, metrics in results.items():
    print(name)
    print(metrics)

FuNet-M
{'accuracy': 0.9880952380952381, 'precision': 1.0, 'recall': 0.9761904761904762, 'f1': 0.9879518072289156, 'auc': 0.993764172335601}


In [31]:
# Saving the model for deployment
# Save in pth format
torch.save(model.state_dict(), "funet_M_full.pth")

print("Funet M model saved successfully!")

Funet M model saved successfully!
